In [1]:
import pandas as pd
import tspaint
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from vscodenb import set_vscode_theme
set_vscode_theme()

In [2]:
s = ! head -n 1000 ../../data/chr20.phased.females.vcf.gz | grep CHROM

ids_in_vcf = pd.Series(sorted(s[0].split()[9:]))

metadata = pd.read_csv('../../data/metadata_with_x_missing.txt', sep=' ')
metadata = metadata.loc[metadata.PGDP_ID.isin(ids_in_vcf)].sort_values('PGDP_ID')

gog_female_ids = metadata.loc[(metadata.Origin == 'Gog Woreda, Gambella region, Ethiopia') & (metadata.Sex == 'F')].PGDP_ID.to_list()
ruaha_female_ids = metadata.loc[(metadata.Origin == 'Ruaha, Tanzania') & (metadata.Sex == 'F')].PGDP_ID.to_list()
mikumi_female_ids = metadata.loc[(metadata.Origin == 'Mikumi, Tanzania') & (metadata.Sex == 'F')].PGDP_ID.to_list()

ref1_ids = gog_female_ids
ref2_ids = mikumi_female_ids
admix_ids = ruaha_female_ids
sample_ids = admix_ids + ref2_ids + ref1_ids

len(sample_ids), len(ref1_ids), len(ref2_ids), len(admix_ids)

(23, 8, 11, 4)

In [3]:
len(ref2_ids + ref1_ids + admix_ids + ref2_ids + admix_ids)

38

In [4]:
labels = {**{n: 0 for n in ref1_ids}, **{n: 1 for n in ref2_ids}}
assert len(labels) == len(ref1_ids) + len(ref2_ids)
region = tspaint.io.subset_data("../../data/chr20.phased.females", start=0, end=2_000_000, samples=sample_ids)

mutation_rate = 1.5e-8
recombination_rate = 1e-8
Ne  = tspaint.io.estimate_ne(region, mutation_rate=mutation_rate)
Ne

57930.94532279315

## tsinfer

In [5]:
tsinfer_ts = tspaint.io.tsinfer(region, 
                                date=True, mutation_rate=1.2e-8
                                )

tsinfer_ts.dump("ts_tsinfer.trees")
print(tsinfer_ts.num_trees, 'trees')

10972 trees


In [ ]:
tsinfer_painting = tspaint.paint(tsinfer_ts, labels)
tsinfer_painting.plot(curves=True)

In [ ]:
qc = tspaint.reference_qc(tsinfer_ts, labels, 
                        #   anchors=set(ref1_ids), 
                          n_jobs=8, progress=True)
soft_refs = qc.soft_refs()
l = qc.summary()
df = pd.DataFrame({k: [d[k] for d in l] for k in l[0]})
df['soft_ref'] = df['ref'].isin(soft_refs)

In [ ]:
df.sort_values(['individual', 'haplotype']).set_index(['individual', 'haplotype'])
#df.sort_values('credibility')
#df.set_index(['individual', 'haplotype'])#.sort_values('is_anchor')

In [ ]:
suspect = qc.summary()[0]['ref']
print('suspect reference:', suspect)
print('flagged foreign tracts:', qc.flagged_tracts(suspect, deadband=0.4))

In [ ]:
# Make foreign_tracts (and all others) take n_jobs arg and make id default to all available (also on slurm)

In [ ]:
ft = tspaint.foreign_tracts(tsinfer_ts, labels, soft_refs, soft_refs=set(soft_refs), min_score=0.4)

In [ ]:
flagged = {r: ft[r] for r in soft_refs if ft[r]}      # references carrying a foreign tract
print(f'{len(flagged)}/{len(soft_refs)} impure references carry a flagged foreign tract')

for r, tracts in list(flagged.items())[:3]:
    print(f'  sample {r}:', [(round(l), round(rr), round(sc, 2), round(rr - l)) for (l, rr, sc) in tracts])

tract_lengths = []
for r, tracts in list(flagged.items()):
    for (l, rr, sc) in tracts:
        # print(f'  sample {r}:', (round(l), round(rr), round(sc, 2), round(rr - l))) 
        tract_lengths.append(rr - l)
plt.hist(tract_lengths, bins=100);

In [ ]:
tsinfer_painting = tspaint.paint(tsinfer_ts, labels, smooth=True, 
                                #  soft_refs=soft_refs, 
                                 n_jobs=8, progress=True)
tsinfer_painting.save("tsinfer_painting.npz")

In [ ]:
tsinfer_painting.segments()

In [ ]:
idx = 5
segments = tsinfer_painting.segments()
lens = [e-s for (s, e, l) in segments[idx] if l == 0]
print(min(lens), max(lens))
plt.hist([np.log10(e-s) for (s, e, l) in segments[idx] if l == 0], bins=30, alpha=0.5) ;
#plt.hist([np.log10(e-s) for (s, e, l) in segments[idx] if l == 1], bins=30, alpha=0.5) ;

In [ ]:
tsinfer_painting.plot()

In [ ]:
tspaint.SegmentTrack(tsinfer_painting.segments(deadband=0.9)).plot()

In [ ]:
#tsinfer_painting = tspaint.Painting.load("tsinfer_painting.npz")
tspaint.SegmentTrack(tsinfer_painting.segments()).plot()

RFmix on data simulated from same tree sequence:

In [ ]:
tsinfer_rfmix_segments = tspaint.compare.rfmix_paint(
    tsinfer_ts,
    labels,
    admix_ids,
    K=2,
    recombination_rate=recombination_rate,
    mutation_rate=mutation_rate
)
with open("tsinfer_rfmix_segments.pkl", "wb") as f:
    pickle.dump(tsinfer_rfmix_segments, f)
tspaint.SegmentTrack(tsinfer_rfmix_segments).plot()     

In [ ]:
dated = tspaint.io.tsinfer(region, date=True, mutation_rate=1.2e-8)   # node ages in generations
rtt   = tspaint.paint(dated, labels, n_jobs=8, progress=True).rate_through_time()          # centers now meaningful

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4))
rtt.plot(ax=ax)
ax.axvline(T_split, color="k", ls="--", lw=1, label=f"true split ({T_split})")
ax.set_title("Cross-ancestry rate through time")
fig.tight_layout()

## Relate

In [ ]:
ts_relate = tspaint.io.relate(region, mutation_rate=mutation_rate, recombination_rate=recombination_rate)
ts_relate.dump("ts_relate.trees")
print(ts_relate.num_trees, 'trees')

In [ ]:
qc = tspaint.reference_qc(ts_relate, labels, 
                        #   anchors=set(ref1_ids), 
                          n_jobs=8, progress=True)
soft_refs = qc.soft_refs()
l = qc.summary()
df = pd.DataFrame({k: [d[k] for d in l] for k in l[0]})
df['soft_ref'] = df['ref'].isin(soft_refs)

In [ ]:
#df.sort_values('credibility')
df = df.sort_values(['individual', 'haplotype']).set_index(['individual', 'haplotype'])
#print(df.to_string())
df

In [ ]:
relate_painting = tspaint.paint(ts_relate, labels, smooth=True, soft_refs=soft_refs, n_jobs=8, progress=True)
relate_painting.save("relate_painting.npz")

In [ ]:
relate_painting.plot()

RFmix on data simulated from same tree sequence:

In [ ]:
relate_rfmix_segments = tspaint.compare.rfmix_paint(
    ts_relate,
    labels,
    admix_ids,
    K=2,
    recombination_rate=recombination_rate,
    mutation_rate=mutation_rate
)
with open("relate_rfmix_segments.pkl", "wb") as f:
    pickle.dump(relate_rfmix_segments, f)
tspaint.SegmentTrack(relate_rfmix_segments).plot()     

## Singer

In [ ]:
# # this samples 250 tree sequences from
# singer_ts = tspaint.io.singer(region, m=mutation_rate, ratio=1, Ne=Ne, n_samples=250, thin=20, burn_in=50)

In [ ]:

# for i, ts in enumerate(singer_ts):
#     singer_ts[i].dump(f"ts_singer_{i}.trees")
#     print(i, ts.num_trees, 'trees')


In [ ]:

# # qc = tspaint.reference_qc(singer_ts, labels, anchors=set(ref1_ids))
# # qc.summary()
# # soft_refs = qc.soft_refs()
# # with open("singer_soft_refs.pkl", "wb") as f:
# #     pickle.dump(soft_refs, f)
# # print(f'soft refs: {", ".join(map(str, soft_refs))}')

# singer_painting = tspaint.paint(singer_ts[::20], # <- this is a subset of the 250 samples, to reduce memory usage
#                                 labels, smooth=True, soft_refs=mikumi_female_ids, n_jobs=8, progress=True)
# singer_painting.save("singer_painting.npz")

In [ ]:
# #painting = tspaint.Painting.load("singer_painting.npz")
# singer_painting.plot() 

## ARGweaver

In [ ]:
# ts_ens_argweaver = tspaint.io.argweaver(region, Ne=Ne, 
#                                         mutation_rate=mutation_rate, recombination_rate=recombination_rate,
#                                         compress=10)
# ts_ens_argweaver_painting = tspaint.paint(ts_ens_argweaver, labels) 

In [ ]:
# ts_ens_argweaver_painting.plot()

## Full chr20

In [ ]:
# %%monitor
# ref1_ids = mikumi_female_ids
# ref2_ids = gog_female_ids
# admixed_ids = ruaha_female_ids
# sample_ids = ruaha_female_ids + ref1_ids + ref2_ids

# labels = {**{n: 0 for n in ref1_ids}, **{n: 1 for n in ref2_ids}}

# region = subset_data("../../data/chr20.phased.females", samples=sample_ids)

# ens = tspaint.io.singer_windowed(
#     # "data/chr20.phased.females",          # ts / VCF / VCF-Zarr / Variants — or a subset_data(...) slice
#     region,
#     window_size=2_000_000, n_jobs=16, 
#     mutation_rate=1.25e-8, recombination_rate=1e-8,
#     # Ne=1e4, 
#     # n_samples=20, thin=10, burn_in=5,
#     seed=42
# )
# # ens -> list of stitched posterior ARGs, same shape as singer(); feed straight to:
# painting = tspaint.paint(ens, labels)